# 08 — Análise de sentimentos nos discursos parlamentares

Extensão analítica: medir o **tom (sentimento)** dos discursos e cruzá-lo com **tema** e
**partido**. Usamos um **modelo de sentimento pronto, multilíngue** (off-the-shelf) — não há
rótulo de sentimento na base, e discurso parlamentar é texto formal. Por isso o resultado é
**exploratório/descritivo** (mesma postura do *domain shift* dos temas), e validamos a
plausibilidade ao final.

- Modelo: `lxyuan/distilbert-base-multilingual-cased-sentiments-student` (positivo/neutro/negativo).
- Score por discurso: `sent_score = P(positivo) − P(negativo)` ∈ [−1, 1].

> Pré-requisitos: `discursos_todos.csv`, `discursos_classificados.csv`, `parlamentares.csv`,
> `resultados_dominio.csv`. Usa `torch`/`transformers` já instalados (modelo baixa do HF Hub).

## 1) Imports e device (GPU com fallback seguro)

In [1]:
import os
os.makedirs("dados", exist_ok=True)
import os, numpy as np, pandas as pd, torch
from transformers import pipeline

def escolher_device():
    """Retorna (indice_device_para_pipeline, msg). Testa um kernel real na GPU:
    alguns builds veem a placa mas nao tem kernel compativel -> cai para CPU."""
    if not torch.cuda.is_available():
        return -1, 'CUDA indisponivel (CPU)'
    try:
        _ = torch.ones(8, device='cuda') @ torch.ones(8, device='cuda')
        torch.cuda.synchronize()
        return 0, f'GPU: {torch.cuda.get_device_name(0)}'
    except Exception as e:
        return -1, f'GPU incompativel ({type(e).__name__}); usando CPU'

dev_idx, msg = escolher_device()        # pipeline usa: 0 = cuda:0, -1 = cpu
print('device', dev_idx, '|', msg)

c:\Users\gutob\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device 0 | GPU: NVIDIA GeForce RTX 5050 Laptop GPU


## 2) Carregar e alinhar os discursos

`discursos_todos.csv` traz a `transcricao`; `discursos_classificados.csv` traz
`temas_previstos`. Os dois arquivos estão **alinhados por linha** (mesma ordem) — juntamos
pelo índice (com `assert` de tamanho).

In [2]:
todos = pd.read_csv('dados/discursos_todos.csv', sep=';', dtype=str)
clf   = pd.read_csv('dados/discursos_classificados.csv', sep=';', dtype=str)
assert len(todos) == len(clf), 'arquivos de discurso desalinhados'

df = todos[['id_deputado', 'nome_deputado', 'data', 'sumario', 'transcricao']].copy()
df['temas_previstos'] = clf['temas_previstos'].values     # alinhado por indice
df['transcricao'] = df['transcricao'].fillna('').str.strip()
df = df[df['transcricao'] != ''].reset_index(drop=True)
print(len(df), 'discursos com texto')

46683 discursos com texto


## 3) Inferência de sentimento (todos os discursos)

Pipeline de sentimento com `truncation=True, max_length=512` (a mediana cabe inteira; a
cauda longa é truncada). Pedimos as 3 probabilidades (`top_k=None`) e calculamos o score
contínuo `sent_score = P(positivo) − P(negativo)`.

In [ ]:
MODELO_SENT = 'lxyuan/distilbert-base-multilingual-cased-sentiments-student'
clf_sent = pipeline('sentiment-analysis', model=MODELO_SENT, device=dev_idx,
                    truncation=True, max_length=512, top_k=None, batch_size=32)

textos = df['transcricao'].tolist()
scores, labels = [], []
for i, saida in enumerate(clf_sent(textos)):           # itera em streaming (batches internos)
    p = {d['label'].lower(): d['score'] for d in saida}
    s = p.get('positive', 0.0) - p.get('negative', 0.0)
    scores.append(s)
    labels.append(max(p, key=p.get))
    if (i + 1) % 5000 == 0:
        print(f'  {i+1}/{len(textos)} discursos')

df['sent_score'] = np.array(scores, dtype=float)
df['sent_label'] = labels
print('pronto. distribuicao de rotulos:')
print(df['sent_label'].value_counts())

In [ ]:
# salva a base de sentimento (sem o texto integral; inclui os temas previstos do discurso)
cols = ['id_deputado', 'nome_deputado', 'data', 'temas_previstos', 'sent_label', 'sent_score']
df[cols].to_csv('dados/discursos_sentimento.csv', index=False, encoding='utf-8')
print('Salvo: dados/discursos_sentimento.csv |', len(df), 'linhas')
print('sent_score: min %.3f / media %.3f / max %.3f' % (df['sent_score'].min(),
      df['sent_score'].mean(), df['sent_score'].max()))

## 4) Sentimento por TEMA

Explodimos `temas_previstos` (`|`) e restringimos aos **temas confiáveis** do *domain shift*
(`resultados_dominio.csv`, `f1_discurso >= 0.50`) — só interpretamos a fundo onde o rótulo de
tema é razoável. Média de `sent_score` por tema.

In [ ]:
dom = pd.read_csv('dados/resultados_dominio.csv')
CORTE = 0.50
temas_confiaveis = dom.loc[dom['f1_discurso'] >= CORTE, 'tema'].tolist()

de = df.assign(tema=df['temas_previstos'].fillna('').str.split('|')).explode('tema')
de = de[de['tema'].isin(temas_confiaveis)]
por_tema = (de.groupby('tema')['sent_score']
            .agg(['mean', 'count']).sort_values('mean'))
print(por_tema.round(3))

In [ ]:
import matplotlib.pyplot as plt
os.makedirs('figuras', exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 0.42 * len(por_tema) + 1))
cores = ['#c0392b' if v < 0 else '#27ae60' for v in por_tema['mean']]
ax.barh(range(len(por_tema)), por_tema['mean'], color=cores)
ax.set_yticks(range(len(por_tema)))
ax.set_yticklabels([t[:34] for t in por_tema.index], fontsize=9)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('sentimento medio  (P(positivo) - P(negativo))')
ax.set_title('Tom medio dos discursos por tema (temas confiaveis)')
plt.tight_layout()
plt.savefig('figuras/sentimento_por_tema.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: figuras/sentimento_por_tema.png')

## 5) Sentimento por PARTIDO

Juntamos com `parlamentares.csv` (partido/UF) e tiramos a média de `sent_score` por partido,
restrita aos ~12 partidos com mais deputados (evita siglas com pouquíssimos discursos).
Leitura possível: governo × oposição (oposição tende a tom mais crítico).

In [ ]:
parl = pd.read_csv('dados/parlamentares.csv', sep=';', dtype=str).rename(columns={'id': 'id_deputado'})
dfp = df.merge(parl[['id_deputado', 'siglaPartido']], on='id_deputado', how='left')

grandes = parl['siglaPartido'].value_counts().head(12).index.tolist()
por_part = (dfp[dfp['siglaPartido'].isin(grandes)]
            .groupby('siglaPartido')['sent_score']
            .agg(['mean', 'count']).sort_values('mean'))
print(por_part.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 0.42 * len(por_part) + 1))
cores = ['#c0392b' if v < 0 else '#27ae60' for v in por_part['mean']]
ax.barh(range(len(por_part)), por_part['mean'], color=cores)
ax.set_yticks(range(len(por_part)))
ax.set_yticklabels(por_part.index, fontsize=9)
ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('sentimento medio  (P(positivo) - P(negativo))')
ax.set_title('Tom medio dos discursos por partido (12 maiores)')
plt.tight_layout()
plt.savefig('figuras/sentimento_por_partido.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: figuras/sentimento_por_partido.png')

## 6) Sentimento por TEMA × PARTIDO (heatmap)

Cruzamento das duas dimensões: média de `sent_score` por (tema confiável, partido grande).
**Verde = tom mais positivo; vermelho = mais negativo.** Linhas ordenadas pelo tom médio geral
do tema (do mais negativo ao mais positivo).

In [ ]:
# media de sent_score por (tema, partido): so temas confiaveis (de) e partidos grandes
dep = de.merge(parl[['id_deputado', 'siglaPartido']], on='id_deputado', how='left')
dep = dep[dep['siglaPartido'].isin(grandes)]
M = dep.pivot_table(index='tema', columns='siglaPartido', values='sent_score', aggfunc='mean')
M = M.reindex(por_tema.index)                      # ordena temas pelo tom medio geral

fig, ax = plt.subplots(figsize=(1.0 * len(M.columns) + 3, 0.5 * len(M.index) + 2))
vmax = np.nanmax(np.abs(M.values)) or 0.01
im = ax.imshow(M.values, cmap='RdYlGn', vmin=-vmax, vmax=vmax, aspect='auto')
ax.set_xticks(range(len(M.columns))); ax.set_xticklabels(M.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(M.index))); ax.set_yticklabels([t[:26] for t in M.index], fontsize=8)
ax.set_title('Sentimento medio por tema x partido (verde: positivo; vermelho: negativo)')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig('figuras/sentimento_tema_partido.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: figuras/sentimento_tema_partido.png')

## 7) Validação de face

Os discursos mais positivos e mais negativos fazem sentido? (olhamos o `sumario`).

In [ ]:
def amostra(sub, titulo):
    print('=' * 80); print(titulo)
    for _, r in sub.iterrows():
        s = str(r['sumario'])[:90].replace(chr(10), ' ')
        print(f"  [{r['sent_score']:+.2f}] {r['nome_deputado'][:24]:<24} | {s}")

amostra(df.sort_values('sent_score', ascending=False).head(6), 'MAIS POSITIVOS')
amostra(df.sort_values('sent_score').head(6), 'MAIS NEGATIVOS')

## 8) Conclusão e ressalvas (para o relatório)

- Análise **exploratória/descritiva**, **não causal**.
- Sentimento vem de um **modelo genérico multilíngue** aplicado a **texto formal/retórico**
  (parlamentar) — há *domain mismatch*; o `sent_score` mede um tom aproximado
  (**negativo ≈ crítico/denúncia**, **positivo ≈ celebratório/homenagem**), não opinião fina.
- Cruzamos só com **temas confiáveis** (mesmo critério do *domain shift*).
- Entregáveis: `discursos_sentimento.csv`, `figuras/sentimento_por_tema.png`,
  `figuras/sentimento_por_partido.png`, `figuras/sentimento_tema_partido.png`.